In [13]:
import torch
import pandas as pd
import numpy as np
import random
import os
import scipy.special

## Load Data & Pre-Processing

In [14]:
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

train.head()

,ID,age,gender,tenure,frequent,payment_interval,subscription_type,contract_length,after_interaction,support_needs
0,TRAIN_00000,54.0,F,47.0,22.0,8.0,member,90,25.0,0
1,TRAIN_00001,30.0,M,16.0,15.0,5.0,vip,360,23.0,0
2,TRAIN_00002,29.0,M,8.0,30.0,21.0,plus,30,21.0,0
3,TRAIN_00003,38.0,F,38.0,23.0,10.0,vip,90,6.0,0
4,TRAIN_00004,25.0,F,52.0,3.0,17.0,member,30,1.0,2


In [15]:
train['gender'] = train['gender'].map({'F':0, "M": 1})
test['gender'] = test['gender'].map({'F':0, "M": 1})

train['subscription_type'] = train['subscription_type'].map({'member' : 0, 'plus' : 1, 'vip' : 2})
test['subscription_type'] = test['subscription_type'].map({'member' : 0, 'plus' : 1, 'vip' : 2})

In [47]:
x_train = train.drop(columns =['ID', 'support_needs'])
x_test = test.drop(columns = ['ID'])

y_train = train['support_needs']
y_train_tensor = torch.tensor(y_train, dtype=torch.long)
y_train_onehot = torch.nn.functional.one_hot(y_train_tensor, num_classes=3)
print(y_train_onehot)

tensor([[1, 0, 0],
        [1, 0, 0],
        [1, 0, 0],
        ...,
        [1, 0, 0],
        [1, 0, 0],
        [0, 0, 1]])


In [48]:
categorical_cols = ['gender', 'subscription_type']
numerical_cols = [col for col in x_train.columns if col not in categorical_cols]

# 1-1. 수치형 데이터 분리
x_train_numerical = x_train[numerical_cols]
x_test_numerical = x_test[numerical_cols]

# 1-2. 범주형 데이터 분리
x_train_categorical = x_train[categorical_cols]
x_test_categorical = x_test[categorical_cols]

# 2-1. train 데이터 기준으로 평균, 표준편차 계산
mean = x_train_numerical.mean()
std = x_train_numerical.std()

# 2-2. 수치형 데이터 표준화
x_train_numerical_scaled = (x_train_numerical - mean) / std
x_test_numerical_scaled = (x_test_numerical - mean) / std

In [49]:
# 3. 범주형 데이터 원-핫 인코딩
x_train_categorical_encoded = pd.get_dummies(x_train_categorical, columns=categorical_cols)
x_test_categorical_encoded = pd.get_dummies(x_test_categorical, columns=categorical_cols)

# 원-핫 인코딩 후 열 이름이 달라지므로 train/test 데이터의 열을 동일하게 맞춰줍니다.
train_cols = x_train_categorical_encoded.columns
test_cols = x_test_categorical_encoded.columns

missing_in_test = set(train_cols) - set(test_cols)
for c in missing_in_test:
    x_test_categorical_encoded[c] = 0

missing_in_train = set(test_cols) - set(train_cols)
for c in missing_in_train:
    x_train_categorical_encoded[c] = 0

x_test_categorical_encoded = x_test_categorical_encoded[train_cols] # 순서 맞추기

In [50]:
# 4-1. 수치형 + 범주형 데이터프레임 결합
x_train_final_df = pd.concat([x_train_numerical_scaled, x_train_categorical_encoded], axis=1)
x_test_final_df = pd.concat([x_test_numerical_scaled, x_test_categorical_encoded], axis=1)

# 4-2. 최종 데이터프레임을 토치 텐서로 변환
x_train_tensor = torch.tensor(x_train_final_df.values, dtype=torch.float32)
x_test_tensor = torch.tensor(x_test_final_df.values, dtype=torch.float32)

# 최종 feature 개수 확인
print(f"최종 학습 데이터의 shape: {x_train_tensor.shape}")
print(f"최종 테스트 데이터의 shape: {x_test_tensor.shape}")

최종 학습 데이터의 shape: torch.Size([30858, 11])
최종 테스트 데이터의 shape: torch.Size([13225, 11])


## Train

In [64]:
# 1. 새로운 활성화 함수 및 미분 함수 정의
def leaky_relu(x, alpha=0.01):
    """LeakyReLU 활성화 함수"""
    return np.where(x > 0, x, x * alpha)

def leaky_relu_derivative(x, alpha=0.01):
    """LeakyReLU의 미분"""
    return np.where(x > 0, 1, alpha)

def softmax(x):
    """Softmax 활성화 함수 (수치적으로 안정적인 버전)"""
    # 아주 큰 값이 입력되어도 overflow가 발생하지 않도록 최댓값을 빼줍니다.
    e_x = np.exp(x - np.max(x, axis=0))
    return e_x / e_x.sum(axis=0)

In [65]:
#neural network class definition
class neuralNetwork:

    #initialize the neural network
    def __init__(self, inputnodes, hiddennodes, outputnodes, learningrate):
        #set number of nodes in each input, hidden, output layer
        self.inodes = inputnodes
        self.hnodes = hiddennodes
        self.onodes = outputnodes

        self.wih = np.random.normal(0.0, pow(self.hnodes, - 0.5),(self.hnodes, self.inodes))
        self.who = np.random.normal(0.0, pow(self.onodes, - 0.5),(self.onodes, self.hnodes))

        #learning rate
        self.lr = learningrate

        # 각 층에 맞는 활성화 함수 지정
        self.hidden_activation = leaky_relu
        self.hidden_derivative = leaky_relu_derivative
        self.output_activation = softmax
        pass

    #신경망 학습시키기
    def train(self, inputs_list, targets_list):
      #입력 리스트를 2차원의 행렬로 변환
        inputs = np.array(inputs_list, ndmin=2).T
      targets = np.array(targets_list, ndmin=2).T
# 순전파
        hidden_inputs = np.dot(self.wih, inputs)
        hidden_outputs = self.hidden_activation(hidden_inputs)

      final_inputs = np.dot(self.who, hidden_outputs)
        final_outputs = self.output_activation(final_inputs)

# 역전파
      #출력 계층의 오차는 (실제 값 - 계산 값)
      output_errors = targets - final_outputs
      #은닉 계층의 오차는 가중치에 의해 나뉜 계층의 오차들을 재조합해 계산
      hidden_errors = np.dot(self.who.T, output_errors)
# 가중치 업데이트
      #은닉 계층과 출력 계층 간의 가중치 업데이트
      self.who += self.lr * np.dot(output_errors, np.transpose(hidden_outputs))

      #입력 계층과 은닉 계층 간의 가중치 업데이트(LeakyReLU 미분 적용)
      self.wih += self.lr * np.dot((hidden_errors * self.hidden_derivative(hidden_outputs)), np.transpose(inputs))
      pass


    #query the neural network
    def query(self, inputs_list):
      inputs = np.array(inputs_list, ndmin=2).T

        # ★ 변경점 6: train과 동일하게 순전파 로직 수정
        hidden_inputs = np.dot(self.wih, inputs)
        hidden_outputs = self.hidden_activation(hidden_inputs)

        final_inputs = np.dot(self.who, hidden_outputs)
        final_outputs = self.output_activation(final_inputs)

        return final_outputs

IndentationError: unexpected indent (4062258292.py, line 30)

In [74]:
#입력, 은닉, 출력 노드의 수
input_nodes = 11
hidden_nodes = 10
output_nodes = 3

#학습률은 0.3
learning_rate = 0.001

#인경망의 인스턴스(instance)를 생성
n = neuralNetwork(input_nodes, hidden_nodes, output_nodes, learning_rate)

In [75]:
epochs = 100
for e in range(epochs):
    for i in range(len(x_train_tensor)):
        # torch.Tensor -> numpy 변환
        inputs = x_train_tensor[i].numpy()
        targets = y_train_onehot[i].numpy()   # y_train도 torch.Tensor라면 변환 필요
        n.train(inputs, targets)

    if e % 10 == 0:
        print(f"epoch: {e}")

epoch: 0
epoch: 10
epoch: 20
epoch: 30
epoch: 40
epoch: 50
epoch: 60
epoch: 70
epoch: 80
epoch: 90


## Prediction

In [76]:
submission_csv = pd.read_csv('sample_submission.csv')
submission=submission_csv['support_needs']
print(submission[0])

0


In [78]:
import numpy as np
import pandas as pd

preds = []
for i in range(len(x_test_tensor)):
    # 표준화된 x_test_tensor 사용
    inputs = x_test_tensor[i].numpy() 
    outputs = n.query(inputs)
    label = np.argmax(outputs)
    preds.append(label)

# DataFrame에 예측값 저장
submission = pd.read_csv('sample_submission.csv')
submission['support_needs'] = preds
submission.to_csv('baseline_submit.csv', index=False)

print("저장 완료: baseline_submit.csv")
print("input_nodes",input_nodes, "hidden_nodes",hidden_nodes,"learning_rate",learning_rate)

저장 완료: baseline_submit.csv
input_nodes 11 hidden_nodes 10 learning_rate 0.001
